# Lecture 10

## Public, Protected, Private

In most objected oriented languages, data encapsulation is achieved by enabling classes to declair protected and private data members in addition to the public ones. Quick recap:

* Public members are accessible to everyone
* Private members are only accessible to the class itself
* Protected members are accessiblve to the all instances of the same class (including child classes) 

Python's implementation of data encapsulation is not strictly enforced by the langauge and is mostly a convention. Data members starting with one underscore (`_`) are protected. Data members starting with two underscores (`__`) are private.

The point of these levels is the abstraction barrier. Public members are the interface a class presents to the world, the part you want to keep stable. Private members sit behind the barrier: since only the class's own methods can change them, nobody outside can corrupt, say, the numbers that make up a matrix, and you stay free to reimplement the inside completely as long as the public interface doesn't change. Protected opens that barrier to your children, but not to the outside world.

The class below is deliberately silly. Its job is to let us pry inside and see how Python actually implements these rules, because none of it is magic.

Consider the following example:

In [1]:
class parent:
    def __init__(self):
        self.public="I'm Public"
        self._protected="I'm Protected"
        self.__private="I'm Private"
        
    def set_private_parent(self,v):
        self.__private=v
        
    def get_private_parent(self):
        return self.__private
        
class child(parent):
    def __init__(self, init_parent=True):
        if init_parent:
            super(child,self).__init__()
    
    def set_public(self,v):
        self.public=v
        
    def set_protected(self,v):
        self._protected=v

    def set_private(self,v):
        self.__private=v
        
    def get_public(self):
        return self.public
        
    def get_protected(self):
        return self._protected

    def get_private(self):
        return self.__private


Lets see what we can do with the private attributes:

In [2]:
# Create an instance of parent:
my_parent = parent()
print(my_parent.__private)

AttributeError: 'parent' object has no attribute '__private'

Notice what the error says: not that the attribute is private, but that the object has no attribute by that name at all. It gives nothing away about what's inside, and, as a closer look below will show, it is literally true.

We can't access the private attributes from outside of the class. It should be the same with the protected:

In [3]:
print(my_parent._protected)

I'm Protected


In [4]:
my_parent._protected = 'abc'
print(my_parent._protected)

abc


But its not! We can access protected data!

Finally, lets check out public attributes:

In [5]:
print(my_parent.public)
my_parent.public = 5
print(my_parent.public)

I'm Public
5


Note that we can set attributes of the class instead of the instance. 

In [6]:
parent.public = 22
print(parent.public)
print(my_parent.public)

22
5


But it doesn't make sense to do so. 

It's worth being clear about what that did. Data members created in `__init__` don't exist until the constructor runs, which is what gives each instance its own copy. `parent.public = 22` touches no instance: it attaches a new attribute to the class itself, visible to every instance that doesn't have its own. `my_parent` still prints 5 because it has a `public` of its own, created by its constructor and then set to 5, which hides the one on the class. Extending a class after the fact like this is possible, but it isn't good practice, and a few cells below you'll see it bite.

With private data, the accessors decide what the outside world may do with it: leave out the getter and nobody outside can know what's there; leave out the setter and the value can't be changed from outside.

The proper way to access/change a private attribute is using the accessors set setters provided by the class:

In [7]:
my_parent.get_private_parent()
my_parent.set_private_parent(42)
my_parent.get_private_parent()

42

Note that because we declared the data members in the parent constructor, we have to make sure that the child calls the parent constructor.

In [8]:
child_instance = child(init_parent=False)
child_instance.public

22

We set this data member earlier, that's why its value is 22. It should have been "I'm public", but we didn't call the constructor of the parent class. 

Having skipped its parent's constructor, the instance never got a `public` of its own, so the lookup fell through to the value put on the class by hand.

If everything is done correctly:

In [9]:
child_instance = child()
child_instance.public

"I'm Public"

Even though we wrote accessors (setters and getters), we can directly access and set the data:

In [10]:
child_instance = child()
print(child_instance.public)
child_instance.public="Changed Public"
print(child_instance.public)

I'm Public
Changed Public


Of couse the accessors also work:

In [11]:
child_instance = child()
print(child_instance.get_public())
child_instance.set_public("Changed Public")
print(child_instance.get_public())

I'm Public
Changed Public


How about the protected?

In [12]:
print(child_instance._protected)
child_instance._protected="Changed Protected"
print(child_instance._protected)

I'm Protected
Changed Protected


In [13]:
child_instance = child()
print(child_instance.get_protected())
child_instance.set_protected("Changed Protected")
print(child_instance.get_protected())

I'm Protected
Changed Protected


So there isn't any difference between public and protected in python. We can just adopt the convention that data members starting with a single underscore will be only accessed via accessors. 

How about private?

In [14]:
print(child_instance.__private)
child_instance.__private="Changed Private"
print(child_instance.__private)

AttributeError: 'child' object has no attribute '__private'

It appears that we finally have some protection. A closer look shows how its done:

In [15]:
dir(child_instance)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_parent__private',
 '_protected',
 'get_private',
 'get_private_parent',
 'get_protected',
 'get_public',
 'public',
 'set_private',
 'set_private_parent',
 'set_protected',
 'set_public']

Note that instead of a data member `__private` we have a data member `_parent__private`. All python does is to replace anything that has the pattern `<class_name>.__<data_name>` to `<class_name>._<class_name>__<data_name>`. So in fact, we can still change this data member and there is no real protection:

In [16]:
print(child_instance._parent__private)
child_instance._parent__private="Changed Private"
print(child_instance._parent__private)

I'm Private
Changed Private


Just to make sure we understand, here what the parent class looks like:

In [17]:
parent_instance=parent()
dir(parent_instance)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_parent__private',
 '_protected',
 'get_private_parent',
 'public',
 'set_private_parent']

Note that a child cannot change a parent's private data:

In [18]:
child_instance = child()
print(child_instance.get_private())
child_instance.set_private("Changed Private")
print(child_instance.get_private())

AttributeError: 'child' object has no attribute '_child__private'

In [19]:
child_instance = child()
print(child_instance.get_private_parent())
child_instance.set_private_parent("Changed Private")
print(child_instance.get_private_parent())

I'm Private
Changed Private


Why don't I get an error in the following case? 

In [20]:
child_instance = child()
print(child_instance.get_private_parent())
child_instance.set_private("Changed Private")
print(child_instance.get_private_parent())

I'm Private
I'm Private


Note that the code doesn't work as intended... why? 

See if you can figure it out by looking at:

In [21]:
dir(child_instance)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_child__private',
 '_parent__private',
 '_protected',
 'get_private',
 'get_private_parent',
 'get_protected',
 'get_public',
 'public',
 'set_private',
 'set_private_parent',
 'set_protected',
 'set_public']

### What happened

The listing now has two private names, `_parent__private` and `_child__private`. The renaming uses the class the name is written in, so inside a method of `child`, `self.__private` becomes `self._child__private`. `set_private` didn't fail: it quietly created and set a new member belonging to the child, while `get_private_parent` kept reading `_parent__private`, which never changed. No crash, no warning, and the value you meant to change stayed the same. That is a real bug people hit, and it is hard to find without knowing the rule.

Underneath, an object and its class are little more than dictionaries of names and values, some of those values functions. The object-oriented syntax is syntactic sugar: it looks and works like object-oriented programming, but who may touch what is carried by conventions and renaming, not enforced. That's fine while everyone follows the conventions, but it makes Python a poor choice when something really must be protected, such as a library others derive from but must not change. For data science, the flexibility is usually worth more. When something behaves strangely, read the error message, look inside with `dir`, and experiment until you understand what the language did. It isn't hiding anything from you.

## Data Serialization

Data serialization refers to the process of converting data (usually in memory) that may have complex structure (e.g. a tree), into a linear sequence that can be use to reconstitute the original data structure. Such a sequence can be stored in a file or transmitted over a network. 

Some vocabulary. Data in memory is *transient*: turn off the computer and it's gone. Data on disk is *persistent*. The two representations of the same information needn't look alike: in memory you want it uncompressed and quick to reach, while on disk you may care more about compactness, or about how fast it can be read and written.

In memory, objects can point at each other along complicated, even circular, paths, with no beginning and no end. Serializing is like describing a picture to someone so they can draw it: the picture isn't linear, but your description has to be, and they rebuild the picture from it. Why not just copy the block of memory into a file? Its layout depends on the machine that produced it, and the receiver may be a completely different kind of computer.

For example consider the following "simple" data structure:

In [22]:
# Simple Data Type

data_dict = { "A": 1, 
              "B": "Foo"}

### Python `repr`

The python `repr` method of build-ins and classes you implement can be used as a means of serialization. Take any python built in and you can see it's string representation, which is essentially a string of python code that can evaluates to the object:

In [23]:
repr(data_dict)

"{'A': 1, 'B': 'Foo'}"

This representation can be easily written to a file:

In [24]:
with open('file.py',"w") as f: 
    f.write(repr(data_dict))

In [25]:
!cat file.py

{'A': 1, 'B': 'Foo'}

And reconstituted by evaluating the contents of the file:

In [26]:
with open('file.py', 'r') as f: 
    data_dict_reloaded = eval(f.read())

data_dict_reloaded

{'A': 1, 'B': 'Foo'}

The string `eval` works on can come from a file, as above, or be built by the program itself, so Python code can write Python code and evaluate it: powerful, and dangerous for the same reason. Only `eval` text whose source you trust.

Note that `eval` uses the python interpreter to execute python expressions stored in strings:

In [27]:
eval("print('Hello World')")

Hello World


In [28]:
x=eval("1+1")
x

2

### YAML

A Python expression in a file only helps a reader that speaks Python. Formats that any language can read came up with the web, where data pulled from a database on one server has to be shipped to another program, possibly in another language, in a form both sides agree on. The Python packages for these formats all look alike: a dump that writes an object out in the format, and a load that reads it back and rebuilds the object.

There are other standard formats for storing simple data types. For example YAML:

In [29]:
import yaml
yaml.dump(data_dict)

'A: 1\nB: Foo\n'

In [30]:
with open('file.yaml',"w") as f: 
    f.write(yaml.dump(data_dict))

In [31]:
!cat file.yaml

A: 1
B: Foo


In [32]:
!ls 

Lecture.10.a.ipynb M.pickle           file.json          file.yaml
Lecture.10.b.ipynb M_list.pickle      file.pickle
M.npy              Scores.csv         file.py


In [33]:
with open('file.yaml', 'r') as f: 
    data_dict_reloaded = yaml.safe_load(f.read())

data_dict_reloaded

{'A': 1, 'B': 'Foo'}

### JSON

For exchanging data, JSON is the format you'll meet most often: browsers exchange information in it, and in data science it is often how metadata comes. For a photo, the pixel values are the data; when and where it was taken is metadata, and that kind of descriptive information usually travels as JSON. JSON text also looks almost exactly like a Python dictionary written out, apart from small differences such as `null` for `None` and `true`/`false` for `True`/`False`. To read it, though, use the `json` package, not `eval`. JSON is also how data gets handed to large language models: the context a model works from, and the structured data it reads and writes, are very often JSON.

[JSON](https://www.json.org/json-en.html) is commonly used to transmit data on the web:

In [34]:
import json
json.dumps(data_dict)

'{"A": 1, "B": "Foo"}'

In [35]:
with open('file.json',"w") as f: 
    json.dump(data_dict,f)

In [36]:
!cat file.json

{"A": 1, "B": "Foo"}

In [37]:
with open('file.json', 'r') as f: 
    data_dict_reloaded = json.load(f)

data_dict_reloaded

{'A': 1, 'B': 'Foo'}

### XML

XML is another format commonly used for storing data. It allows a bit more structure and there are python tools for creating XML representations of data, but it's a bit more complicated than the example above, so we'll skip it for now.

XML uses the tagged markup of HTML, wrapping each piece of data in opening and closing tags. It is verbose and awkward to write by hand, and JSON is more common. All of these formats store data (dictionaries, lists, numbers, strings), not code.

### pickle

[pickle](https://docs.python.org/3/library/pickle.html) is python's method of serialing objects. Some advantages are that it is a binary format, so it is more compact, and that it can store full python objects, not just simple built-ins. Lets look at the [pickle documentation](https://docs.python.org/3/library/pickle.html) first.

The documentation's comparison with JSON is worth reading. JSON is text, human readable, and understood well beyond Python; pickle is none of those. It is Python-specific, and its format has gone through several protocol versions, so a file written in one environment can fail to load under a different Python version.

That settles what pickle is for. It isn't for sharing data with other people, or for keeping data long term. It is for checkpointing your own work: if a notebook builds something complicated that you'd rather not recompute, and you want it back in another notebook or session in the same environment, pickle is the right tool. If the information is going to someone else, use a standard format. Note also the `"wb"` and `"rb"` below: the `b` means binary, since pickle doesn't write text.

One more thing to be careful of. Pickle follows every reference an object holds, so a class that refers to itself, or objects that refer to each other in a circle, need care: following the references leads straight back to where it started. Pickle has machinery for this, but it is a complication to be aware of when you pickle anything more than simple data.

Here is an example:

In [38]:
import pickle
pickle.dumps(data_dict,protocol=2)

b'\x80\x02}q\x00(X\x01\x00\x00\x00Aq\x01K\x01X\x01\x00\x00\x00Bq\x02X\x03\x00\x00\x00Fooq\x03u.'

In [39]:
with open('file.pickle',"wb") as f: 
    pickle.dump(data_dict,f)

In [40]:
!cat file.pickle

��       }�(�A�K�B��Foo�u.

In [41]:
with open('file.pickle', 'rb') as f: 
    data_dict_reloaded = pickle.load(f)

data_dict_reloaded

{'A': 1, 'B': 'Foo'}

## Python classes

Imagine you have data stored in a python object:

In [42]:
# Instance of a python class with data

class data_class:
    def __init__(self):
        self._data = dict()
    
    def add(self,key,value):
        self._data[key]=value
        
    def get(self,key):
        return self._data[key]
    
    def __repr__(self):
        return self._data.__repr__()

data_class_instance = data_class()
data_class_instance.add("A",1)
data_class_instance.add("B","Foo")

print("Value of A:", data_class_instance.get("A"))
print("Value of B:", data_class_instance.get("B"))

Value of A: 1
Value of B: Foo


Since we implemented `__repr__`, I should be able to store the data using `repr`:

In [43]:
with open('file.py',"w") as f: 
    f.write(repr(data_class_instance))

In [44]:
!cat file.py

{'A': 1, 'B': 'Foo'}

In [45]:
!ls -l

total 47272
-rwxr--r--@ 1 afarbin  staff    36258 Feb 17 13:40 Lecture.10.a.ipynb
-rwxr-xr-x@ 1 afarbin  staff   134721 Feb 17 11:36 Lecture.10.b.ipynb
-rw-r--r--@ 1 afarbin  staff  8000128 Feb  7 10:37 M.npy
-rw-r--r--@ 1 afarbin  staff  8000163 Feb  7 10:37 M.pickle
-rw-r--r--@ 1 afarbin  staff  8000163 Feb  7 10:37 M_list.pickle
-rwxr--r--@ 1 afarbin  staff     2428 Feb  7 10:37 Scores.csv
-rw-r--r--@ 1 afarbin  staff       20 Feb 17 13:32 file.json
-rw-r--r--@ 1 afarbin  staff       32 Feb 17 13:38 file.pickle
-rw-r--r--@ 1 afarbin  staff       20 Feb 17 13:41 file.py
-rw-r--r--@ 1 afarbin  staff       12 Feb 17 13:30 file.yaml


In [46]:
with open('file.py', 'r') as f: 
    data_class_instance_reloaded = eval(f.read())

data_class_instance_reloaded

{'A': 1, 'B': 'Foo'}

But what I get back is not the original object reconstituted, but a dictionary holding the data:

In [47]:
type(data_class_instance_reloaded)

dict

In [48]:
data_class_instance_reloaded.add("C",2)

AttributeError: 'dict' object has no attribute 'add'

In [49]:
data_class_instance_reloaded

{'A': 1, 'B': 'Foo'}

The mistake is in `__repr__`, and it's there on purpose, to make a point: it returns the representation of the dictionary inside the class. What got written was a prescription for a dictionary, not for the `data_class` wrapping it, so a dictionary came back, and a dictionary has no `add` method.

The fix is for `__repr__` to return the code you would type to construct the object: the class name, called with its data, the data itself written using its own representation. Store the constructor along with the data, not just the data. This is the approach Lab 4 suggests for saving a drawing to a file and loading it back; the example at the end of that lab has exactly this shape.

We can modify the class to work.

In [50]:
# Instance of a python class with data

class data_class:
    def __init__(self,d=None):
        if d:
            self._data=d
        else:
            self._data = dict()
    
    def add(self,key,value):
        self._data[key]=value
        
    def get(self,key):
        return self._data[key]
    
    def __repr__(self):
        return "data_class("+self._data.__repr__()+")"

data_class_instance = data_class()
data_class_instance.add("A",1)
data_class_instance.add("B","Foo")

print("Value of A:", data_class_instance.get("A"))
print("Value of B:", data_class_instance.get("B"))

Value of A: 1
Value of B: Foo


In [51]:
with open('file.py',"w") as f: 
    f.write(repr(data_class_instance))

with open('file.py', 'r') as f: 
    data_class_instance_reloaded = eval(f.read())

data_class_instance_reloaded

data_class({'A': 1, 'B': 'Foo'})

In [52]:
!cat file.py

data_class({'A': 1, 'B': 'Foo'})

In [53]:
data_class_instance_reloaded.add("C",2)

### pickle

That works, but if the dictionary held millions of entries, the file would be all of that data spelled out as text inside `data_class(...)`. Pickle does better: hand it the instance, and you get the instance back, methods and all.

"Methods and all" needs one caution. Pickle doesn't copy a class's code into the file; it records which class the object belongs to (the `__main__.data_class` below is the class defined in this notebook), and loading needs that code to be available. An object from a library, such as a numpy array, needs the library at load time, and if the library has changed enough since the file was written, the file may not load.

Pickle allows me to store the object:

In [54]:
with open('file.pickle',"wb") as f: 
    pickle.dump(data_class_instance,f)

In [55]:
with open('file.pickle', 'rb') as f: 
    data_class_instance_reloaded = pickle.load(f)

data_class_instance_reloaded

data_class({'A': 1, 'B': 'Foo'})

In [56]:
type(data_class_instance_reloaded)

__main__.data_class

In [57]:
data_class_instance_reloaded.add("C",2)

## Storing Multiple Objects into Pickle

Use a dictionary.

In [58]:
data_class_instance_2 = data_class()
data_class_instance_2.add("C",2)
data_class_instance_2.add("D","Bar")

In [59]:
with open('file.pickle',"wb") as f: 
    pickle.dump({"my_class":data_class_instance,
                 "my_class_2":data_class_instance_2},
                f)

In [60]:
with open('file.pickle', 'rb') as f: 
    loaded_data = pickle.load(f)

data_class_instance_reloaded = loaded_data["my_class"]
data_class_instance_reloaded_2 = loaded_data["my_class_2"]

## Pickling Data

Now a large block of numbers: a 1000 by 1000 random matrix, saved with pickle and with `np.save`, numpy's own standard `.npy` format. Compare the sizes of `M.pickle` and `M.npy` in the listing: essentially identical. A class can overload the methods pickle calls to serialize it, the same way we've overloaded other built-ins, and a numpy array does, writing the same raw block of numbers that `np.save` writes. Still, to send an array to someone else, send the `.npy` file, a standard format, not the pickle.

In [61]:
import numpy as np
M = np.random.random((1000,1000))

In [62]:
with open('M.pickle',"wb") as f: 
    pickle.dump(M, f)

In [63]:
np.save("M.npy",M)

In [64]:
!ls -lh

total 48040
-rwxr--r--@ 1 afarbin  staff    38K Feb 17 13:42 Lecture.10.a.ipynb
-rwxr-xr-x@ 1 afarbin  staff   132K Feb 17 11:36 Lecture.10.b.ipynb
-rw-r--r--@ 1 afarbin  staff   7.6M Feb 17 13:44 M.npy
-rw-r--r--@ 1 afarbin  staff   7.6M Feb 17 13:44 M.pickle
-rw-r--r--@ 1 afarbin  staff   7.6M Feb  7 10:37 M_list.pickle
-rwxr--r--@ 1 afarbin  staff   2.4K Feb  7 10:37 Scores.csv
-rw-r--r--@ 1 afarbin  staff    20B Feb 17 13:32 file.json
-rw-r--r--@ 1 afarbin  staff   132B Feb 17 13:43 file.pickle
-rw-r--r--@ 1 afarbin  staff    32B Feb 17 13:42 file.py
-rw-r--r--@ 1 afarbin  staff    12B Feb 17 13:30 file.yaml


In [65]:
!ls -l

total 48040
-rwxr--r--@ 1 afarbin  staff    40360 Feb 17 13:44 Lecture.10.a.ipynb
-rwxr-xr-x@ 1 afarbin  staff   134721 Feb 17 11:36 Lecture.10.b.ipynb
-rw-r--r--@ 1 afarbin  staff  8000128 Feb 17 13:44 M.npy
-rw-r--r--@ 1 afarbin  staff  8000163 Feb 17 13:44 M.pickle
-rw-r--r--@ 1 afarbin  staff  8000163 Feb  7 10:37 M_list.pickle
-rwxr--r--@ 1 afarbin  staff     2428 Feb  7 10:37 Scores.csv
-rw-r--r--@ 1 afarbin  staff       20 Feb 17 13:32 file.json
-rw-r--r--@ 1 afarbin  staff      132 Feb 17 13:43 file.pickle
-rw-r--r--@ 1 afarbin  staff       32 Feb 17 13:42 file.py
-rw-r--r--@ 1 afarbin  staff       12 Feb 17 13:30 file.yaml


In [66]:
M_list=M.tolist()

In [67]:
with open('M_list.pickle',"wb") as f: 
    pickle.dump(M, f)

In [68]:
!ls -lh

total 48040
-rwxr--r--@ 1 afarbin  staff    39K Feb 17 13:44 Lecture.10.a.ipynb
-rwxr-xr-x@ 1 afarbin  staff   132K Feb 17 11:36 Lecture.10.b.ipynb
-rw-r--r--@ 1 afarbin  staff   7.6M Feb 17 13:44 M.npy
-rw-r--r--@ 1 afarbin  staff   7.6M Feb 17 13:44 M.pickle
-rw-r--r--@ 1 afarbin  staff   7.6M Feb 17 13:45 M_list.pickle
-rwxr--r--@ 1 afarbin  staff   2.4K Feb  7 10:37 Scores.csv
-rw-r--r--@ 1 afarbin  staff    20B Feb 17 13:32 file.json
-rw-r--r--@ 1 afarbin  staff   132B Feb 17 13:43 file.pickle
-rw-r--r--@ 1 afarbin  staff    32B Feb 17 13:42 file.py
-rw-r--r--@ 1 afarbin  staff    12B Feb 17 13:30 file.yaml


In [ ]:
!ls -l

In [ ]:
!rm *.pickle *.yaml *.json file.py

## What's next

Part b, below, turns to the format real data most often arrives in, comma-separated values. It builds a CSV reader by hand, working out how to give each field the right type, how to represent the table in memory, and why reading the file should stay separate from that representation, before showing pandas doing the same job. It then looks closely at `__getitem__` and indexing, the mechanism Lab 5's `matrix` class needs for both `M[i][j]` and `M[i,j]`, and ends with lazy evaluation.


## Reading Tabular Data

Why write a CSV (Camma Separated Values) reader by hand when pandas reads a CSV file in a single line? Real data very often arrives as a comma-separated file, and doing the reading by hand once shows what that line does: taking data off the disk, interpreting it, and keeping it in memory in a form someone had to choose. Then you'll see why pandas is built the way it is.

The file is real: per-question scores from an earlier offering of a course, with no names, exported from a Google Sheet to find the questions that gave students trouble.

We will start with playing around with a data file... lets make sure it's there:

In [1]:
!ls 

Lecture.10.a.ipynb Lecture.10.b.ipynb Lecture.10.ipynb   Scores.csv


In [7]:
!cat Scores.csv

## Comma Separated Values (CSV) File Format

The simplest and most common file format for storing data is called Comma Separated Values (CSV). Generally a CSV file represents a table, with the top row (first line of the file) consisting of the labels of the columns (separated by commas).  Each column keeps a different feature or field. For example for student data, the first column could be the name, second the ID, third major, etc. After the first line, each row hold the data for one data point or example. In the case of student data, each row could correspond to one student.

### Reading CSV Files

There are lots of libraries for reading CSV files into memory. Before we start using them, lets write our own. We'll need two things:

* Means of reading and interpreting the file.
* A representation of the read data in memory.

Keep those two jobs separate in your head. A table is a table: a CSV file is one way of writing it out, and an Excel file is another. What we want is to load a table from whatever format it's in, work with it in memory in one consistent representation, and write it back out in whatever format we like. The format on disk is chosen for convenience or efficient storage; the representation in memory should be chosen for access, whatever makes the data easy to work with.

Lets recall one of the ways in python to read a file.

In [8]:
f=open("Scores.csv","r")

first_line = f.readline()
print(first_line)

line = f.readline()
while line:
    print(line)
    line = f.readline()

f.close()

l1_n,l1_1,12_n,l2_1,l2_2,l2_3,l2_4,l2_5,l2_6,l2_7,l3_n,l3_1,l3_2,l3_3,l3_4,l3_5,l3_6,l3_7,l3_8,l3_9,l3_10,l3_11,l3_12,l3_13,l3_14,l4_n,l4_1,l4_2,l4_3,l4_4,l4_5,l4_6,l4_7,l4_8,l4_9,l4_10,l4_11,q1_n,q1_1,e1_n,e1_1,e1_2,e1_3,e1_4,e1_5,e1_6,e1_7,e1_8,e1_9,e1_10,e1_11,e1_12,e1_13,e1_14,e1_15

1,10,7,10,10,9.5,9,10,10,9.5,14,10,10,10,10,10,10,9,10,3,0,3,3,5,2,11,0,0,0,0,0,0,0,0,0,0,0,1,0,15,9,9,10,5,5,0,0,10,10,10,10,0,10,5,10

1,10,7,0,10,9.5,0,10,10,0,14,10,10,10,10,0,0,0,0,0,0,0,0,0,0,11,10,10,10,10,5,3,0,3,10,7,0,1,9.5,15,9,9,10,5,10,0,9,9,9,9,9,10,5,0,0

1,10,7,10,10,0,10,10,10,10,14,10,6,10,0,0,0,0,0,0,0,0,0,0,0,11,10,10,0,7,0,0,0,0,0,0,0,1,9.5,15,9,9,10,9,5,9,7,9,10,10,10,5,10,5,0

1,10,7,10,10,10,9.5,10,10,9.5,14,10,10,10,10,0,0,0,0,0,0,0,0,0,0,11,10,10,10,10,3,3,0,0,5,0,0,1,10,15,9,9,10,0,10,0,7,5,9,9,9,0,0,0,0

1,10,7,10,10,5,9.5,10,10,9.5,14,5,9,9,10,7,10,10,10,10,7,10,3,5,10,11,0,0,0,0,0,0,0,0,0,0,0,1,10,15,9,9,9,8,7,10,0,9,10,9,10,9,5,0,0

1,10,7,10,10,3,7,10,10,9,14,10,10,10,10

We successfully dumped the contents of the file, but we didn't 

* interpret it... each line is just a string, not a camma separated list of keys or values. 
* or store it into memory... we just dumped it into the screen.

Let's start on properly interpreting the first line, which is special:

In [9]:
f=open("Scores.csv","r")
first_line = f.readline()
print(first_line.split(","))
f.close()

['l1_n', 'l1_1', '12_n', 'l2_1', 'l2_2', 'l2_3', 'l2_4', 'l2_5', 'l2_6', 'l2_7', 'l3_n', 'l3_1', 'l3_2', 'l3_3', 'l3_4', 'l3_5', 'l3_6', 'l3_7', 'l3_8', 'l3_9', 'l3_10', 'l3_11', 'l3_12', 'l3_13', 'l3_14', 'l4_n', 'l4_1', 'l4_2', 'l4_3', 'l4_4', 'l4_5', 'l4_6', 'l4_7', 'l4_8', 'l4_9', 'l4_10', 'l4_11', 'q1_n', 'q1_1', 'e1_n', 'e1_1', 'e1_2', 'e1_3', 'e1_4', 'e1_5', 'e1_6', 'e1_7', 'e1_8', 'e1_9', 'e1_10', 'e1_11', 'e1_12', 'e1_13', 'e1_14', 'e1_15\n']


It appears that each line ends with `\n`. Here's how we can remove these newlines.

In [10]:
f=open("Scores.csv","r")
first_line = f.readline().rstrip()
print(first_line.split(","))
f.close()

['l1_n', 'l1_1', '12_n', 'l2_1', 'l2_2', 'l2_3', 'l2_4', 'l2_5', 'l2_6', 'l2_7', 'l3_n', 'l3_1', 'l3_2', 'l3_3', 'l3_4', 'l3_5', 'l3_6', 'l3_7', 'l3_8', 'l3_9', 'l3_10', 'l3_11', 'l3_12', 'l3_13', 'l3_14', 'l4_n', 'l4_1', 'l4_2', 'l4_3', 'l4_4', 'l4_5', 'l4_6', 'l4_7', 'l4_8', 'l4_9', 'l4_10', 'l4_11', 'q1_n', 'q1_1', 'e1_n', 'e1_1', 'e1_2', 'e1_3', 'e1_4', 'e1_5', 'e1_6', 'e1_7', 'e1_8', 'e1_9', 'e1_10', 'e1_11', 'e1_12', 'e1_13', 'e1_14', 'e1_15']


Finally lets store the first line, which is a list of the column names:

In [11]:
f=open("Scores.csv","r")
first_line = f.readline().rstrip()
keys=first_line.split(",")
f.close()

In [12]:
keys

['l1_n',
 'l1_1',
 '12_n',
 'l2_1',
 'l2_2',
 'l2_3',
 'l2_4',
 'l2_5',
 'l2_6',
 'l2_7',
 'l3_n',
 'l3_1',
 'l3_2',
 'l3_3',
 'l3_4',
 'l3_5',
 'l3_6',
 'l3_7',
 'l3_8',
 'l3_9',
 'l3_10',
 'l3_11',
 'l3_12',
 'l3_13',
 'l3_14',
 'l4_n',
 'l4_1',
 'l4_2',
 'l4_3',
 'l4_4',
 'l4_5',
 'l4_6',
 'l4_7',
 'l4_8',
 'l4_9',
 'l4_10',
 'l4_11',
 'q1_n',
 'q1_1',
 'e1_n',
 'e1_1',
 'e1_2',
 'e1_3',
 'e1_4',
 'e1_5',
 'e1_6',
 'e1_7',
 'e1_8',
 'e1_9',
 'e1_10',
 'e1_11',
 'e1_12',
 'e1_13',
 'e1_14',
 'e1_15']

Now lets read the rest of the file in a similar fashion:

In [13]:
f=open("Scores.csv","r")
first_line = f.readline().rstrip()
keys=first_line.split(",")

data=list()

line = f.readline().rstrip()
while line:
    data.append(line.split(","))
    line = f.readline().rstrip()

f.close()

In [14]:
data

[['1',
  '10',
  '7',
  '10',
  '10',
  '9.5',
  '9',
  '10',
  '10',
  '9.5',
  '14',
  '10',
  '10',
  '10',
  '10',
  '10',
  '10',
  '9',
  '10',
  '3',
  '0',
  '3',
  '3',
  '5',
  '2',
  '11',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '1',
  '0',
  '15',
  '9',
  '9',
  '10',
  '5',
  '5',
  '0',
  '0',
  '10',
  '10',
  '10',
  '10',
  '0',
  '10',
  '5',
  '10'],
 ['1',
  '10',
  '7',
  '0',
  '10',
  '9.5',
  '0',
  '10',
  '10',
  '0',
  '14',
  '10',
  '10',
  '10',
  '10',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '11',
  '10',
  '10',
  '10',
  '10',
  '5',
  '3',
  '0',
  '3',
  '10',
  '7',
  '0',
  '1',
  '9.5',
  '15',
  '9',
  '9',
  '10',
  '5',
  '10',
  '0',
  '9',
  '9',
  '9',
  '9',
  '9',
  '10',
  '5',
  '0',
  '0'],
 ['1',
  '10',
  '7',
  '10',
  '10',
  '0',
  '10',
  '10',
  '10',
  '10',
  '14',
  '10',
  '6',
  '10',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
  '0',
 

In [15]:
len(data)

16

In [16]:
len(data[3])

55

We have everything in memory now, how do we retrieve it?

To associate specific keys to column numbers, we can do the following

In [17]:
keys.index("l4_1")

26

So the 10th student "l4_1" grade is:

In [18]:
data[10][keys.index("l4_1")]

'0'

Note that it's still a string... not a number:

In [19]:
type(data[10][keys.index("l4_1")])

str

### Representing a table with Python Built-ins

Lets quickly think of several ways of representing a table in memory:
1. A list of lists (i.e. a 2-D array or matrix) and a dictionary, where each element of the "matrix" corresponds to a specific grade for a specific student and the dictionary maps the name of the column to the column index. For example `data[5][column_names["l1_5"]]` corresponds to the 6th student's grade on lab 3 question 5.

1. A list of dictionaries, where each element of the list is corresponds to a row of the table and the dictionaries are keyed by the column name. For example `data[5]["l3_5"]` corresponds to the 6th student's grade on lab 3 question 5.


Note that both methods above store the data, indexing the row first, aka *row-wise*. If this was a large dataset and the expected access pattern (how we plan to use the data), required looking at all information in a row at a time, then such an implementation would be most efficient. If we instead wanted to look the values a single column for all rows, then we be better off with a *column-wise* representation:

3. A dictionary of lists, where each element of the dictionary corresponds to a column of data and the lists contain the data in that column. For example `data["l3_5"][5]` corresponds to the 6th student's grade on lab 3 question 5.

A list of dictionaries is nicer to use than a list of lists, but it isn't free: every row's dictionary stores its own copy of every key, which is extra memory spent purely for convenience.

Why would row-wise versus column-wise matter? When the CPU reads a value, it copies a small block of memory around it into its very fast cache, so values you read one after another are fastest when they sit next to each other. At the size of this file it hardly matters. But in a particle physics experiment a single collision can be hundreds of megabytes of data, effectively one row, while an analysis may need just one number from each collision to fill a histogram; stored row-wise, getting those few megabytes means reading through everything. Experiments like that deliberately store data differently on disk than they hold it in memory, because the access patterns differ. When you decide how to store data, think about how you're going to read it back.

For example, we can easily choose number 2 for our storage. We'll go through it slowly in a bit.

In [20]:
f=open("Scores.csv","r")
first_line = f.readline().rstrip()
keys=first_line.split(",")

data=list()

line = f.readline().rstrip()
while line:
    data.append(dict(zip(keys,line.split(","))))
    line = f.readline().rstrip()

f.close()
data[5]["l3_5"]

'0'

## Building a CSV Reader

We have the basics down, but we have more things to consider:
* We have written some example code, we should now write something that is general and we could use in different instances. 
* The fields can be different types: strings, numbers (integer or floating point). We should store the fields as the correct data type.
* Still need to figure out a nice way to store the data in memory.
* We might want to also be able to write out CSV files.

We have some options on how to proceed:
   1. **Function reader / python built-in data representation**- We could write a CSV reader function `my_data = read_csv(filename)` that given the filename of a CSV file, reads the data and returns it as a standard python data object. As we discussed above, there are various suitable representations for `my_data`, so we'll either have to pick one or provide some options to allow for other ones.
   1. **One class for reading and storage**- Instead of a CSV reader function, we could create a CSV reader class. It will be instantiated with a CSV filename `my_file = csv_file(filename)`, so each instance would be uniquely connected to a specific file. It'll read the data into some representation that is kept private. We provide accessor methods to get to retrieve specific parts of the data, or the whole data as standard python data.
   1. **Separate classes for reading and storing**- We can separate the concepts of a CSV reader and how we store the data. In this way, we could write other readers (e.g. Excel file reader) that would still use the same data storage. This is the option we are going to explore below.

The second design, one class for reading and storage, looks natural, since object-oriented programming puts data and code together, but it's the design to avoid. If the reading code is married to the representation in memory, the data can only be reached through that code, and wherever the data goes, the code has to go too.

The same principle applies to algorithms. A matrix or tensor class can carry accessors and helpers, but the algorithms that turn one tensor into another should live outside it, so you can run different algorithms on the same data and compare them: putting code and data together doesn't mean putting algorithms and data together. The gradebook built in Lectures 11 and 12 is designed around exactly this split.

### File Handling Class

The design below uses the virtual-method pattern from Lecture 8. The base class does what every format shares: it checks the file's extension, then calls `_readfile`, which it leaves unimplemented. `CSVHandler` inherits all of that and supplies only `_readfile`, so supporting another format, say Excel or JSON, means inheriting from `DataFileHandler` and implementing just that one method. These classes store no data; they organize *how* we read files.

Consider the following implementation of a file reader that leaves room for supporting other file formats:

In [21]:
class DataFileHandler:
    def __init__(self,extensions):
        self.__extensions=extensions
        
    def check_extension(self,filename):
        file_extension=filename.split(".")[-1]
        return file_extension in self.__extensions

    def _readfile(self,filename):
        raise NotImplementedError    
        
    def readfile(self,filename,check_extension=True):
        if not check_extension or self.check_extension(filename):
            return self._readfile(filename)        
        else:
            print("Error: filename {} does not match acceptable extensions.".format(filename))
    
    def _writefile(self,filename,data):
        raise NotImplementedError
        
    def writefile(self,filename,data):
        return self._writefile(filename,data)
        
        
class CSVHandler(DataFileHandler):
    def __init__(self):
        #super(CSVHandler,self).__init__(["csv","CSV"])
        DataFileHandler.__init__(self,["csv","CSV"])
        
    def _readfile(self,filename):
        f=open(filename,"r")
        first_line = f.readline().rstrip()
        fields=first_line.split(",")

        data=list()

        line = f.readline().rstrip()
        while line:
            data.append(line.split(","))
            line = f.readline().rstrip()

        f.close()
        
        return fields,data
        
    

Note that this implementation doesn't do anything fancy with the data representation. But does allow for multiple handlers to be created to read various types of files.

Lets test:

In [22]:
my_handler=CSVHandler()
fields,data=my_handler.readfile("Scores.csv")

In [23]:
type(data[0][3])

str

In [24]:
data[0][3]

'10'

#### Handling Different Types

When we read a text file, all the content is assumed to be composed of strings. Instead, ideally we would like to interpret the file by looking at each field and selecting an appropriate type.

So we need a way to check if a string can be converted to a number (e.g. float). 

A way to check if a string can be converted to a number is to simply attempt to make it into a float, if it succeeds, then all is well, if not, we'll get an error which we'll catch and handle appropriately.

We could instead write a function that checks the characters one by one, handling decimal points, minus signs and everything else a number can contain. Or we can simply ask Python. In C++, exceptions are expensive, and you wouldn't use them just to test whether a string is a number. In Python they aren't, and attempting the conversion and falling back when it fails is good Python for exactly this job. One caveat: it's a little unsafe, because a value meant to stay a string but written in digits will quietly become a number.

Catch only the error you expect. Catch every error, and a broken program doesn't crash with a message: it just silently does nothing. So first we need to know exactly which error a failed conversion produces.

What is the error we get? Lets induce the error and see what python says:

In [2]:
float("foo")

ValueError: could not convert string to float: 'foo'

`ValueError` is the type of error we'll catch. And when the conversion works, there is no error:

In [3]:
float(5)

5.0

Here's an example of how we can catch the error:

In [4]:
x="5"
y="foo"

try:
    x=float(x)
except ValueError:
    pass

try:
    y=float(y)
except ValueError:
    pass

print("x:",x,type(x))
print("y:",y,type(y))

x: 5.0 <class 'float'>
y: foo <class 'str'>


Lets add this to our `CSVHandler`:

In [1]:
class DataFileHandler:
    def __init__(self,extensions):
        self.__extensions=extensions
        
    def check_extension(self,filename):
        file_extension=filename.split(".")[-1]
        return file_extension in self.__extensions

    def _readfile(self,filename):
        raise NotImplementedError    
        
    def readfile(self,filename,check_extension=True):
        if not check_extension or self.check_extension(filename):
            return self._readfile(filename)        
        else:
            print("Error: filename {} does not match acceptable extensions.".format(filename))
    
    def _writefile(self,filename,data):
        raise NotImplementedError
        
    def writefile(self,filename,data):
        return self._writefile(filename,data)
        
        
class CSVHandler(DataFileHandler):
    def __init__(self):
        #super(CSVHandler,self).__init__(["csv","CSV"])
        DataFileHandler.__init__(self,["csv","CSV"])
        
    def _readfile(self,filename):
        f=open(filename,"r")
        first_line = f.readline().rstrip()
        fields=first_line.split(",")

        data=list()

        line = f.readline().rstrip()
        while line:
            items=line.split(",")
            
            row=list()
            for item in items:
                # Using Error Catching (Exception handling) to test type
                try:
                    d=float(item)
                except ValueError:
                    d=item
                row.append(d)
            
            data.append(row)
            
            line = f.readline().rstrip()

        f.close()
        
        return fields,data
        
    

In [4]:
my_handler=CSVHandler()
fields,data=my_handler.readfile("Scores.csv")

In [5]:
type(data[0][3])

float

### Data Representation with python built-ins

Since we are using a list of lists to contain the data of the CSV file in memory, we have to do a bit of manipulation to find specific fields in the list of lists:

In [6]:
data[10][fields.index("l4_1")]

0.0

As discussed above, we should use a dictionary instead... recall some basics:

In [7]:
foo=dict()
foo["L1"]=1
foo["L2"]=2

foo

{'L1': 1, 'L2': 2}

In [8]:
foo["L1"]

1

In [9]:
dict([ ("L1",1), ("L2",2) ])

{'L1': 1, 'L2': 2}

So in principle, we can then take every row and convert it easily to a dictionary:

In [10]:
first_row=dict(list(zip(fields,data[0])))

In [11]:
first_row

{'l1_n': 1.0,
 'l1_1': 10.0,
 '12_n': 7.0,
 'l2_1': 10.0,
 'l2_2': 10.0,
 'l2_3': 9.5,
 'l2_4': 9.0,
 'l2_5': 10.0,
 'l2_6': 10.0,
 'l2_7': 9.5,
 'l3_n': 14.0,
 'l3_1': 10.0,
 'l3_2': 10.0,
 'l3_3': 10.0,
 'l3_4': 10.0,
 'l3_5': 10.0,
 'l3_6': 10.0,
 'l3_7': 9.0,
 'l3_8': 10.0,
 'l3_9': 3.0,
 'l3_10': 0.0,
 'l3_11': 3.0,
 'l3_12': 3.0,
 'l3_13': 5.0,
 'l3_14': 2.0,
 'l4_n': 11.0,
 'l4_1': 0.0,
 'l4_2': 0.0,
 'l4_3': 0.0,
 'l4_4': 0.0,
 'l4_5': 0.0,
 'l4_6': 0.0,
 'l4_7': 0.0,
 'l4_8': 0.0,
 'l4_9': 0.0,
 'l4_10': 0.0,
 'l4_11': 0.0,
 'q1_n': 1.0,
 'q1_1': 0.0,
 'e1_n': 15.0,
 'e1_1': 9.0,
 'e1_2': 9.0,
 'e1_3': 10.0,
 'e1_4': 5.0,
 'e1_5': 5.0,
 'e1_6': 0.0,
 'e1_7': 0.0,
 'e1_8': 10.0,
 'e1_9': 10.0,
 'e1_10': 10.0,
 'e1_11': 10.0,
 'e1_12': 0.0,
 'e1_13': 10.0,
 'e1_14': 5.0,
 'e1_15': 10.0}

In [12]:
first_row["l4_1"]

0.0

And then store the dictionary for each row in a list:

In [13]:
new_data=list()

for row in data:
    new_data.append(dict(list(zip(fields,row))))

In [14]:
new_data

[{'l1_n': 1.0,
  'l1_1': 10.0,
  '12_n': 7.0,
  'l2_1': 10.0,
  'l2_2': 10.0,
  'l2_3': 9.5,
  'l2_4': 9.0,
  'l2_5': 10.0,
  'l2_6': 10.0,
  'l2_7': 9.5,
  'l3_n': 14.0,
  'l3_1': 10.0,
  'l3_2': 10.0,
  'l3_3': 10.0,
  'l3_4': 10.0,
  'l3_5': 10.0,
  'l3_6': 10.0,
  'l3_7': 9.0,
  'l3_8': 10.0,
  'l3_9': 3.0,
  'l3_10': 0.0,
  'l3_11': 3.0,
  'l3_12': 3.0,
  'l3_13': 5.0,
  'l3_14': 2.0,
  'l4_n': 11.0,
  'l4_1': 0.0,
  'l4_2': 0.0,
  'l4_3': 0.0,
  'l4_4': 0.0,
  'l4_5': 0.0,
  'l4_6': 0.0,
  'l4_7': 0.0,
  'l4_8': 0.0,
  'l4_9': 0.0,
  'l4_10': 0.0,
  'l4_11': 0.0,
  'q1_n': 1.0,
  'q1_1': 0.0,
  'e1_n': 15.0,
  'e1_1': 9.0,
  'e1_2': 9.0,
  'e1_3': 10.0,
  'e1_4': 5.0,
  'e1_5': 5.0,
  'e1_6': 0.0,
  'e1_7': 0.0,
  'e1_8': 10.0,
  'e1_9': 10.0,
  'e1_10': 10.0,
  'e1_11': 10.0,
  'e1_12': 0.0,
  'e1_13': 10.0,
  'e1_14': 5.0,
  'e1_15': 10.0},
 {'l1_n': 1.0,
  'l1_1': 10.0,
  '12_n': 7.0,
  'l2_1': 0.0,
  'l2_2': 10.0,
  'l2_3': 9.5,
  'l2_4': 0.0,
  'l2_5': 10.0,
  'l2_6': 10.0,


In [15]:
new_data[10]["l3_4"]

10.0

### Data Representation in Custom Class

Our representation of a table as a list of dictionaries isn't the most efficient, but it is convenient. 

Lets try a different approach to storing the data that is custom made for storing tables with rows of data. 


In [16]:
class DataRow:
    def __init__(self,fields,data):
        self.__fields=fields
        self.__data=data
        
    def __getitem__(self,key):
        return self.__data[self.__fields.index(key)]


class Data:
    def __init__(self):
        self.__fields=list()
        self.__data=list()
        
    def set_fields(self,fields):
        self.__fields=fields
        
    def add_data_point(self,data_point):
        if isinstance(data_point,list):
            if len(data_point) == len(self.__fields):
                self.__data.append(DataRow(self.__fields,data_point))
            else:
                print("Expected {} fields, got {} fields.".format(len(self.__fields),len(fields)))
        else:
            print("Data Point must be given as a list.")

    def add_data_points(self,data_points):
        for data_point in data_points:
            self.add_data_point(data_point)
            
    def fields(self):
        return self.__fields
    
    def __getitem__(self,key):
        return self.__data[key]

    def __str__(self):
        return self.__fields

In [17]:
my_data=Data()
my_data.set_fields(fields)
my_data.add_data_points(data)

In [18]:
my_data[12]["l3_4"]

10.0

In [19]:
my_data[12]

A table is a collection of rows, so we wrote one class for a row and one for the collection. `DataRow` is effectively a dictionary wrapped in a class we control, and `Data` keeps its rows private, so the only way to add one is `add_data_point`, which first checks the number of values. In `my_data[12]["l3_4"]` above, the first bracket calls `Data`'s `__getitem__`, which hands back a `DataRow`, and the second calls that row's `__getitem__`. Pay attention to this two-layer pattern: it's the one you'll need for the matrix class in Lab 5.

Now lets put it all together:

In [22]:
class DataFileHandler:
    def __init__(self,extensions):
        self.__extensions=extensions
        
    def check_extension(self,filename):
        file_extension=filename.split(".")[-1]
        return file_extension in self.__extensions

    def _readfile(self,filename):
        raise NotImplementedError    
        
    def readfile(self,filename,check_extension=True):
        if not check_extension or self.check_extension(filename):
            return self._readfile(filename)        
        else:
            print("Error: filename {} does not match acceptable extensions.".format(filename))
    
    def _writefile(self,filename,data):
        raise NotImplementedError
        
    def writefile(self,filename,data):
        return self._writefile(filename,data)
        
        
class CSVHandler(DataFileHandler):
    def __init__(self):
        #super(CSVHandler,self).__init__(["csv","CSV"])
        DataFileHandler.__init__(self,["csv","CSV"])
        
    def _readfile(self,filename):
        f=open(filename,"r")
        first_line = f.readline().rstrip()
        fields=first_line.split(",")

        data=list()

        line = f.readline().rstrip()
        while line:
            items=line.split(",")
            
            row=list()
            for item in items:
                try:
                    d=float(item)
                except ValueError:
                    d=item
                row.append(d)
            
            data.append(row)
            
            line = f.readline().rstrip()

        f.close()
        
        my_data=Data()
        my_data.set_fields(fields)
        my_data.add_data_points(data)
        
        return my_data
        
    

In [23]:
my_handler=CSVHandler()
my_data=my_handler.readfile("Scores.csv")

In [24]:
my_data[10]["l4_1"]

0.0

In [25]:
type(my_data)

__main__.Data

## Pandas

What we just build is very similar to Pandas...

In [26]:
import pandas as pd
Data=pd.read_csv("Scores.csv")

In [27]:
list(filter(lambda x: "read" in x, dir(pd)))

['read_clipboard',
 'read_csv',
 'read_excel',
 'read_feather',
 'read_fwf',
 'read_gbq',
 'read_hdf',
 'read_html',
 'read_json',
 'read_orc',
 'read_parquet',
 'read_pickle',
 'read_sas',
 'read_spss',
 'read_sql',
 'read_sql_query',
 'read_sql_table',
 'read_stata',
 'read_table',
 'read_xml']

In [28]:
type(Data)

pandas.core.frame.DataFrame

In [29]:
Data

,l1_n,l1_1,12_n,l2_1,l2_2,l2_3,l2_4,l2_5,l2_6,l2_7,...,e1_6,e1_7,e1_8,e1_9,e1_10,e1_11,e1_12,e1_13,e1_14,e1_15
0,1,10,7,10,10,9.5,9.0,10,10,9.5,...,0,0,10,10,10,10,0,10,5,10
1,1,10,7,0,10,9.5,0.0,10,10,0.0,...,0,9,9,9,9,9,10,5,0,0
2,1,10,7,10,10,0.0,10.0,10,10,10.0,...,9,7,9,10,10,10,5,10,5,0
3,1,10,7,10,10,10.0,9.5,10,10,9.5,...,0,7,5,9,9,9,0,0,0,0
4,1,10,7,10,10,5.0,9.5,10,10,9.5,...,10,0,9,10,9,10,9,5,0,0
5,1,10,7,10,10,3.0,7.0,10,10,9.0,...,10,10,10,10,10,10,10,9,8,2
6,1,10,7,10,10,3.0,9.5,10,10,9.5,...,9,0,0,10,10,9,5,10,8,0
7,1,10,7,10,10,0.0,5.0,10,10,9.5,...,0,0,0,0,0,0,0,0,0,0
8,1,10,7,0,0,0.0,0.0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
9,1,10,7,10,10,10.0,9.5,10,10,9.5,...,10,7,0,9,9,9,0,5,0,0


In [30]:
dir(Data)

['T',
 '_AXIS_LEN',
 '_AXIS_ORDERS',
 '_AXIS_TO_AXIS_NUMBER',
 '_HANDLED_TYPES',
 '__abs__',
 '__add__',
 '__and__',
 '__annotations__',
 '__array__',
 '__array_priority__',
 '__array_ufunc__',
 '__array_wrap__',
 '__bool__',
 '__class__',
 '__contains__',
 '__copy__',
 '__dataframe__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__divmod__',
 '__doc__',
 '__eq__',
 '__finalize__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__ifloordiv__',
 '__imod__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__nonzero__',
 '__or__',
 '__pos__',
 '__pow__',
 '__radd__',
 '__rand__',
 '__rdivmod__',
 '__reduce__',
 '__reduce_ex_

In [31]:
Data[Data["l4_2"]==10]

,l1_n,l1_1,12_n,l2_1,l2_2,l2_3,l2_4,l2_5,l2_6,l2_7,...,e1_6,e1_7,e1_8,e1_9,e1_10,e1_11,e1_12,e1_13,e1_14,e1_15
1,1,10,7,0,10,9.5,0.0,10,10,0.0,...,0,9,9,9,9,9,10,5,0,0
2,1,10,7,10,10,0.0,10.0,10,10,10.0,...,9,7,9,10,10,10,5,10,5,0
3,1,10,7,10,10,10.0,9.5,10,10,9.5,...,0,7,5,9,9,9,0,0,0,0
5,1,10,7,10,10,3.0,7.0,10,10,9.0,...,10,10,10,10,10,10,10,9,8,2
6,1,10,7,10,10,3.0,9.5,10,10,9.5,...,9,0,0,10,10,9,5,10,8,0
7,1,10,7,10,10,0.0,5.0,10,10,9.5,...,0,0,0,0,0,0,0,0,0,0
9,1,10,7,10,10,10.0,9.5,10,10,9.5,...,10,7,0,9,9,9,0,5,0,0
10,1,10,7,10,10,9.5,0.0,10,10,0.0,...,9,7,9,10,10,10,10,10,0,0
12,1,10,7,10,10,9.5,9.5,10,10,9.5,...,9,7,9,10,10,10,10,0,0,0
13,1,10,7,10,10,0.0,0.0,10,10,7.0,...,10,9,9,10,10,10,10,10,5,10


In [32]:
Data.columns

Index(['l1_n', 'l1_1', '12_n', 'l2_1', 'l2_2', 'l2_3', 'l2_4', 'l2_5', 'l2_6',
       'l2_7', 'l3_n', 'l3_1', 'l3_2', 'l3_3', 'l3_4', 'l3_5', 'l3_6', 'l3_7',
       'l3_8', 'l3_9', 'l3_10', 'l3_11', 'l3_12', 'l3_13', 'l3_14', 'l4_n',
       'l4_1', 'l4_2', 'l4_3', 'l4_4', 'l4_5', 'l4_6', 'l4_7', 'l4_8', 'l4_9',
       'l4_10', 'l4_11', 'q1_n', 'q1_1', 'e1_n', 'e1_1', 'e1_2', 'e1_3',
       'e1_4', 'e1_5', 'e1_6', 'e1_7', 'e1_8', 'e1_9', 'e1_10', 'e1_11',
       'e1_12', 'e1_13', 'e1_14', 'e1_15'],
      dtype='object')

pandas does the same job, with a reader function for each format (filtering `dir(pd)` for "read" above lists them), and every one of them gives back the same thing: a `DataFrame`, best thought of as a very nice, shiny container for a table.

Look closely at `Data[Data["l4_2"]==10]`. `==` is overloaded so that comparing a column to a value gives back a column of `True`/`False`, one for each row. `__getitem__` then dispatches on what it receives: a string selects a column, while a list of booleans as long as the number of rows is a mask that selects rows, giving a new `DataFrame` of just the students who got a 10 on that question. None of it is magic, just a `__getitem__` that checks what came in. Doing the same selection on our own `Data` objects would take loops and a good deal more code; picking out a subset of rows is common enough that a good in-memory representation should make it quick. Lecture 11 puts this selection to work cleaning a messier grades file.

The `DataFrame` class is also built exactly the way Part a described. Look through `dir(Data)` and you'll find a long list of protected and private members, internals you're not supposed to touch, alongside the public methods that are the interface for working with tables.

## Indexing

We implemented `__getitem__` in our data classes to enable easy access of our data. Lets take a closer look:

In [35]:
class my_list:
    def __init__(self,a_list):
        self._list=a_list
        
    def __getitem__(self,key):
        print(key)
        pass
        #return self._list[key]
        

We get a lot of nice functionality... but not everything:

In [36]:
obj = my_list([5,5,5])

obj[1]
obj[1,2]
obj[1,2,3]
obj[1:2]

1
(1, 2)
(1, 2, 3)
slice(1, 2, None)


Note slicing results in a `slice` object not the slice of the data. Let's look at `slice` closer:

In [37]:
slice(1,2,3)

slice(1, 2, 3)

In [38]:
dir(slice)

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'indices',
 'start',
 'step',
 'stop']

In [39]:
slice(1,2,3).start

1

In [40]:
obj[:1]

slice(None, 1, None)


We can detect in `__getitem__` when we get a slice object and use it accordingly:

In [41]:
class my_list:
  def __init__(self,a_list):
    self._list=a_list

  def __getitem__(self, key):
    if isinstance(key, slice):
        start = key.start or 0
        stop = key.stop or len(self._list)
        step = key.step or 1        
        return [self._list[i] for i in range(start, stop, step)]
    elif isinstance(key, int):
        return self._list[key]
    elif isinstance(key, tuple):
        raise NotImplementedError
    else:
        raise TypeError 

In [42]:
my_list([1,2,3,4,5,6,7])[2:6:1]

[3, 4, 5, 6]

What if we want to do something more complicated, like handle both `M[i][j]` and `M[i,j]` in the same way?

`Data` and `DataRow` already show how to get `M[i][j]`. For `M[i,j]`, recall from the experiment above that `obj[1,2]` arrives in `__getitem__` as a single tuple, `(1, 2)`. So the outer `__getitem__` checks its key. If it's a tuple, use the first item to find the row and pass the second item to that row's `__getitem__`; if it isn't, just return the row and let the second bracket do the rest. That's what Lab 5 asks of your matrix class.

Notice how we found out what `__getitem__` receives: a throwaway class that prints its key, then a `slice` made by hand and inspected with `dir`. Use the notebook as a sketchpad for small puzzles like this, and don't be afraid to look under the hood.

## Lazy Evaluation

Matrix multiplication can be a time consuming operation. What if we only need a few elements of the result of a matrix multiplication? Can we some how only compute the elements we need? This is where the concept of lazy evaluation can come in handy. We have already seen that python generators are a built-in mechanism for lazy evaluation. Here, we create our own implementation.

Multiplying two $n \times n$ matrices the straightforward way is an $n^3$ operation: make the matrices twice as big and it takes eight times as long, ten times as big and it takes a thousand times as long. Now suppose you need the product of two million-by-million matrices, but will only ever look at a hundred elements of the result, and you don't know in advance which hundred.

Recall the matrix multiplication formula:

 $C=A \cdot B$: $C_{ij} = \sum_{k} A_{ik} B_{kj}$.
 

Take the time to match each index in this formula to a loop in the code below. Reading equations is part of a data scientist's job, so don't skip ahead.

Note that we actually compute every element of the resultant matrix independently, but in a typical implemenation of multiplication, we'll loop over all elements of resultant matrix:

In [5]:
def zero_matrix(m,n):
    return [ [0 for _ in range(m)] for _ in range(n)]

def is_matrix(M):
    if isinstance(M,list):
        row_length=len(M[0])
        for row in M:
            if not row_length==len(row):
                return False
    else:
        False
    return True
        

def matrix_shape(M):
    if is_matrix(M):
        m=len(M)
        n=len(M[0])
        return m,n
    else:
        0,0

def matrix_multiply(M1,M2):
    m1,n1=matrix_shape(M1)
    m2,n2=matrix_shape(M2)
    
    if n1==m2:
        
        M3=zero_matrix(m1,n2)
        
        # Loop over ALL elements of the resultant matrix
        for i in range(m1):
            for j in range(n2):
                # Compute the element
                for k in range(n1):
                    M3[i][j]+=M1[i][k]*M2[k][j]
        return M3
    
    return False

In [44]:
M1 =  [ [ 1, 2 ] , [ 2, 3 ] ]
M2 =  [ [ 1, 2 ] , [ 2, 3 ] ]

matrix_multiply(M1,M2)
    

[[5, 8], [8, 13]]

`is_matrix` exists because a list of lists doesn't constrain anything: each inner list can have a different length, while every row of a matrix must be the same length.

The class below also checks its inputs, with `assert`, which you haven't seen much: if the condition after it is true the code carries on, and if it's false Python stops with an error. It's a quick way of making sure the two matrices can be multiplied before going any further.

Instead we can create a new matrix class that holds the results of a product of two matrices... and only computes the elements it needs.

In [45]:
class lazy_multiplied_matrix:
    def __init__(self,M1,M2):
        m1,n1=matrix_shape(M1)
        m2,n2=matrix_shape(M2)
        
        assert(n1==m2)
        
        self._n1=n1
        
        self._m=m1
        self._n=n2
        
        self._M1=M1
        self._M2=M2

        # By default the resultant matrix will be composed of Nones, 
        # indicating that no element is computed.
        
        self._M3= [ [None for _ in range(self._m)] for _ in range(self._n)]
        
    def __getitem__(self,key):
        if isinstance(key,tuple):
            i,j=key
        else:
            return None

        if self._M3[i][j]:
            return self._M3[i][j]
        else:
            self._M3[i][j]=0.

            for k in range(self._n1):
                self._M3[i][j]+=self._M1[i][k]*self._M2[k][j]
                
            return self._M3[i][j]
        
    def __str__(self):
        return str(self._M3)
    
    __repr__ = __str__
    
                    

In [46]:
M3=lazy_multiplied_matrix(M1,M2)

In [47]:
M3

[[None, None], [None, None]]

In [48]:
M3[1,1]

13.0

In [49]:
M3

[[None, None], [None, 13.0]]

`M3` started out as all `None`s, meaning nothing had been computed. Asking for `M3[1,1]` computed that one element from the formula and stored it, so it doesn't need computing again. (It has to be indexed as `M3[i,j]`: its `__getitem__` gives back `None` for anything that isn't a tuple.) You index this object like a matrix, but it isn't really a representation of a matrix: it's a computation of the product of two matrices that happens to cache its results. Something that looks like data can actually be a computation.

Based on everything presented today, can you imagine how you could implement sparse matrices?

Here is one way to think about it. A sparse matrix might be a million by a million, with only a small fraction of its elements non-zero. Store only the non-zero elements, for example in a dictionary that maps a tuple of indices to a value, and have `__getitem__` give back zero for anything that isn't there. The cost is that every stored element now carries its indices, roughly three numbers for every one, so this saves memory only when fewer than about a third of the elements are non-zero. When you use a library's sparse matrices, it's worth having this notion of how they could work underneath.

## What's next

Lab 5 asks you to build a `matrix` class on nested lists, and the second half of this notebook is its toolkit: two layers of `__getitem__` for `M[i][j]` and `M[i,j]`, detecting a `slice`, the shape checks and the multiplication loop, plus Lecture 9's operator overloading. Lecture 11 then uses pandas to clean a real, messy grades file and compute grades, and starts a gradebook that keeps data classes and algorithm classes apart.